# Fluctuation Diagnostics Analysis

## Overview

This notebook documents VAFT's fluctuation analysis for VEST, from a raw diagnostic
trace to a spectral index. It covers two layers that are deliberately kept apart:

- **diagnostic-specific preparation** — calibration, transfer-function correction and
  windowing, which differ for every diagnostic and live in
  `vaft.machine_mapping` / `vaft.process.magnetics`;
- **generic spectral analysis** — `vaft.process.fluctuation`, which takes a plain
  `(time, data)` pair and knows nothing about where it came from.

The separation is the point. A power spectrum does not care whether its input is a
magnetic field, a soft X-ray brightness or a line-integrated density, but the work
needed to *produce* a physically meaningful input is entirely diagnostic-specific. The
sections below first establish the theory, then show the processing contract, then
apply both to real VEST data from two different diagnostics.

## Objectives

- Explain what each spectral routine computes and which numerical choices it exposes.
- Show how a Mirnov pickup voltage becomes a magnetic-field time series, and why
  skipping that step corrupts the spectral index by exactly 2.
- Demonstrate the input-validation contract, including what it refuses and why.
- Fit power laws, locate spectral breaks in both supported modes, integrate band
  powers, and produce spectrograms — all with caller-chosen intervals.
- Render results through the canonical `vaft.plot` contract and the `vaft.omas`
  adapters.
- Show the same generic API driving a second, unrelated diagnostic without change.

## Expected Inputs

- A diagnostics ODS with native-bandwidth Mirnov voltage
  (`magnetics.b_field_pol_probe.{i}.voltage`), either from `workflow/main` via
  `VAFT_DIAGNOSTICS_ODS` or mapped here from the packaged shot-44740 raw fixture.
- Per-channel calibration constants from the VEST magnetics channel table.
- A packaged VEST soft X-ray digitizer file, for the diagnostic-independence section.

## Expected Outputs

- Calibrated, integrated magnetic-fluctuation time series at native bandwidth.
- Power spectra with fitted spectral indices, fit-quality metrics, spectral breaks and
  band-integrated powers, as typed results.
- Spectrogram views of the same signals.
- Figures written to `VAFT_DOCS_OUTPUT_DIR`.

## Pipeline Context

This notebook sits after raw signal loading and magnetics mapping. Its spectral
products are derived quantities: they are returned as typed objects and deliberately
**not** written back into the ODS, since IMAS has no schema for a fitted spectral
index or a break frequency.

## A note on reference slopes

VAFT ships no spectral-slope constants. Values such as `-5/3` or `-8/3` are not
defaults, not module constants and not automatic classifications anywhere in the
library or in this notebook. Reference guides are supplied by the caller, and what a
fitted slope *means* is the reader's judgement, not the library's.

## Related Notebooks

- `magnetic_diagnostics_processing.ipynb` — upstream calibration and filtering.
- `soft_x_ray_signal_analysis.ipynb` — the SXR diagnostic used here as a second case.
- `plotting_sample_using_vaft_plot_module.ipynb` — the canonical plotting architecture.

## 1. Setup

`vaft.process.fluctuation` is the generic layer; everything imported from
`vaft.process.magnetics` and `vaft.machine_mapping` below is VEST-specific preparation.

In [ ]:
from dataclasses import replace
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
from omas import ODS

import vaft
import vaft.plot as vplot
from vaft.data.resources import data_path
from vaft.machine_mapping.magnetics import (
    magnetics,
    vest_equilibrium_magnetics_channel_definitions,
)
from vaft.machine_mapping.utils import get_path, set_path

# VEST-specific signal preparation.
from vaft.process.magnetics import (
    DEFAULT_VEST_MAGNETICS_PROCESSING,
    toroidal_mode_analysis,
    vest_b_field_pol_probe_legacy,
)

# Diagnostic-independent spectral analysis.
from vaft.process.fluctuation import (
    NONUNIFORM_TOLERANCE,
    analyze_fluctuation_spectrum,
    compute_band_power,
    compute_psd,
    compute_spectrogram,
    find_spectral_break,
    fit_power_law_spectrum,
)

# Canonical view models and renderers.
from vaft.plot.models import PowerSpectrum, ReferenceSlope, Series, Spectrogram
from vaft.plot import render_power_spectrum, render_spectrogram

plt.rcParams["figure.dpi"] = 120
output_dir = Path(os.environ.get("VAFT_DOCS_OUTPUT_DIR", "notebooks/outputs/docs"))
output_dir.mkdir(parents=True, exist_ok=True)

## 2. Load the diagnostics ODS

Set `VAFT_DIAGNOSTICS_ODS` to a local `workflow/main` product when one is available.
Without it, the notebook deterministically maps the packaged shot-44740 raw fixture, so
every number below is reproducible offline.

In [ ]:
configured_ods = os.environ.get("VAFT_DIAGNOSTICS_ODS")
if configured_ods:
    ods_path = Path(configured_ods).expanduser()
    if not ods_path.is_file():
        raise FileNotFoundError(f"VAFT_DIAGNOSTICS_ODS does not exist: {ods_path}")
    ods = vaft.omas.load_omas_json(ods_path, consistency_check=False)
    source = ods_path
else:
    fixture = data_path("legacy/shot_44740.json.gz")
    sample_template = str(fixture.parent / "shot_{shot}.json.gz")
    ods = ODS()
    magnetics(ods, shot=44740, tstart=0.26, tend=0.34, dt=4e-5, raw_source=sample_template)
    source = fixture
source

## 3. Raw Mirnov voltage traces

The MATLAB default rows `15` and `38` are one-based rows in the legacy `md` array. In
this ODS they are zero-based `b_field_pol_probe` channels `14` and `37`.

These are **voltages**, not fields. Section 6.6 explains why that distinction changes
every spectral index computed from them.

In [ ]:
inboard_channel = 14
outboard_channel = 37
time_range = (0.304, 0.330)

fig, ax = vaft.omas.plot_magnetics_time_mirnov_voltage(
    ods,
    channels=[inboard_channel, outboard_channel],
    x_limits=time_range,
    title="Raw Mirnov voltage",
)
fig.savefig(output_dir / "mirnov_raw_voltage.png", dpi=200, bbox_inches="tight")
fig

## 4. MATLAB-parity spectrogram

`vaft.omas.plot_magnetics_spectrogram_mirnov` drives
`vaft.process.magnetics.mirnov_spectrogram`, a manual Hann-window FFT written to
reproduce `vest_mirnov.m` exactly: `window_size=500`, `time_resolution=1`, nominal
`sample_rate=250e3`. It is Mirnov-specific by name and by history, and it is kept for
parity with the legacy MATLAB workflow.

Section 9.3 shows the generic replacement, `compute_spectrogram`, which takes any
scalar signal and expresses its window in seconds rather than samples.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
for axis, channel, label in (
    (axes[0], inboard_channel, "Inboard midplane Mirnov"),
    (axes[1], outboard_channel, "Outboard midplane Mirnov"),
):
    vaft.omas.plot_magnetics_spectrogram_mirnov(
        ods,
        channel=channel,
        time_range=time_range,
        window_size=500,
        time_resolution=1,
        max_frequency=80e3,
        ax=axis,
    )
    axis.set_title(label)
fig.tight_layout()
fig.savefig(output_dir / "mirnov_spectrogram.png", dpi=200, bbox_inches="tight")
fig

## 5. Toroidal phase mode fit

The phase-reference set is stored as voltage-only entries appended to
`magnetics.b_field_pol_probe`. When all four reference channels are available, this
fits wrapped `phase = phase0 - n * phi` lines to the selected time slice; otherwise the
same routine is demonstrated on a synthetic four-channel mode.

`vplot.toroidal_phase_mode_fit` predates the canonical renderer architecture and still
returns its fit result alongside the figure, so it has no drop-in canonical equivalent
yet; it emits a deprecation warning until one exists.

Multi-channel mode analysis is outside the single-channel spectral layer that follows —
cross-spectra, coherence and mode numbers are tracked as separate work.

In [ ]:
def _has_voltage_data(ods, channel):
    try:
        return np.asarray(get_path(ods, f"magnetics.b_field_pol_probe.{channel}.voltage.data")).size > 0
    except Exception:
        return False

phase_channels = [64, 65, 66, 67]
fit_ods = ods
fit_time = 0.3215
fit_time_range = (fit_time - 0.006, fit_time + 0.006)
fit_frequencies = None

if not all(_has_voltage_data(ods, channel) for channel in phase_channels):
    sample_rate = 250_000.0
    fit_time = 0.3215
    time = np.arange(int(0.04 * sample_rate), dtype=float) / sample_rate + 0.305
    angles = np.deg2rad([0.0, 120.0, 180.0, 240.0])
    fit_ods = {}
    for index, angle in enumerate(angles):
        signal = np.sin(2 * np.pi * 26_000.0 * time + 0.5 - 1 * angle)
        signal += 0.7 * np.sin(2 * np.pi * 52_000.0 * time - 0.2 - 2 * angle)
        set_path(fit_ods, f"magnetics.b_field_pol_probe.{index}.name", f"Synthetic phi={np.rad2deg(angle):.0f}")
        set_path(fit_ods, f"magnetics.b_field_pol_probe.{index}.toroidal_angle", angle)
        set_path(fit_ods, f"magnetics.b_field_pol_probe.{index}.voltage.time", time)
        set_path(fit_ods, f"magnetics.b_field_pol_probe.{index}.voltage.data", signal)
    phase_channels = [0, 1, 2, 3]
    fit_time_range = (fit_time - 0.006, fit_time + 0.006)
    fit_frequencies = [26_000.0, 52_000.0]

fig, ax, phase_fit = vplot.toroidal_phase_mode_fit(
    fit_ods,
    center_time=fit_time,
    channels=phase_channels,
    time_range=fit_time_range,
    frequencies=fit_frequencies,
    num_modes=2,
    candidate_n=range(0, 5),
    window_size=500,
    preprocess=True,
    show=False,
    save_path=output_dir / "toroidal_phase_mode_fit.png",
    return_result=True,
)
fig

In [ ]:
[(mode.frequency / 1e3, mode.n, np.rad2deg(mode.rms_error)) for mode in phase_fit.modes]

## 6. Theory

Six ideas underpin `vaft.process.fluctuation`. Each subsection states the definition,
then the numerical choice VAFT makes and the knob it exposes.

### 6.1 Power spectral density and Welch's method

For a stationary signal $x(t)$ the one-sided power spectral density $S_x(f)$
distributes the signal's variance over frequency:

$$\int_0^{f_\mathrm{Nyq}} S_x(f)\,\mathrm{d}f = \operatorname{var}(x).$$

A single periodogram of an $N$-sample record is an unbiased but *inconsistent*
estimator: its variance does not fall as $N$ grows, so it stays about as noisy as the
signal itself no matter how much data you collect. Welch's method fixes this by cutting
the record into $K$ overlapping segments of length `nperseg`, windowing and
transforming each, and averaging the results. The variance of the estimate falls
roughly as $1/K$.

That gives the one trade you cannot avoid:

$$\Delta f = \frac{f_s}{\texttt{nperseg}}, \qquad K \approx \frac{N}{\texttt{nperseg}\,(1-\texttt{overlap})}.$$

A longer segment resolves finer frequency structure — narrow coherent peaks — while a
shorter one averages more segments and gives a smoother estimate, which is what you
want for fitting a broadband power law. `compute_psd` exposes `nperseg`, `noverlap`,
`window` and `detrend` explicitly and picks nothing implicitly, so a published spectral
index can be reproduced from the recorded parameters. Section 8.5 measures the trade on
real data.

Windowing matters because a finite record is a rectangular truncation of an infinite
one, and a rectangle's transform has $-13$ dB sidelobes: a strong low-frequency
component leaks power across the whole band and can masquerade as a broadband floor.
The Hann default trades a slightly wider main lobe for far lower sidelobes.

### 6.2 Power-law spectral index

Broadband fluctuation spectra are often described by a power law over some band,

$$S(f) = A\,f^{\alpha},$$

which is a straight line in log-log coordinates:

$$\log_{10} S = \alpha \log_{10} f + \log_{10} A.$$

`fit_power_law_spectrum` fits that line by least squares in log-log space and returns a
`SpectralFit` carrying `alpha`, the `intercept` $\log_{10}A$, `r_squared`, the slope's
standard error `stderr`, the point count and the log-space `residuals`.

Two deliberate constraints:

- **The fit interval is always explicit.** There is no default `f_range` and no
  automatic range selection, so a reported $\alpha$ always belongs to a band someone
  chose on purpose. Automatic range-picking is how a spectral index becomes an artefact
  of the fitting code rather than a property of the plasma.
- **Fitting happens in log space**, which is what makes the model linear, but it also
  means the fit minimises *relative* error. That is usually what you want for a
  quantity spanning decades, and it is why `r_squared` and `stderr` are reported in log
  space too.

`r_squared` near 1 says the band really is a straight line in log-log; a low value says
the power-law model does not describe that band, whatever $\alpha$ came out. Section
8.1 shows a band with $R^2 = 0.85$ next to one with $R^2 = 0.49$, and the second number
is the honest warning.

### 6.3 Two-regime spectral break

Many spectra steepen at some frequency. The two-regime model is

$$S(f) \propto \begin{cases} f^{\alpha_\mathrm{low}}, & f < f_\mathrm{break}\\[2pt]
f^{\alpha_\mathrm{high}}, & f > f_\mathrm{break}\end{cases}$$

`find_spectral_break` supports two modes, and **never chooses between them for you** —
the mode follows from which argument you pass:

1. **physics-informed** (`break_frequency=`) — you supply the boundary, for instance
   your own ion-cyclotron frequency, and the code fits each side. The result records
   `mode="imposed"`.
2. **data-driven** (`search_range=`) — the code scans candidate splits across an
   interval you specify and reports the best, with `mode="search"`.

The split is kept explicit because the two answer different questions. An imposed break
tests a hypothesis you brought; a searched break describes a feature of the data. VAFT
never identifies either with a plasma scale — that interpretation is yours.

**How candidates are scored.** The search minimises the *total* squared log-space
residual of both segments:

$$\mathrm{SSR}(f_b) = \sum_{f<f_b} r^2 + \sum_{f>f_b} r^2.$$

The obvious alternative — maximising a point-weighted $R^2$ — is biased. Every
candidate partitions the same set of points, so the total residual is directly
comparable across candidates; a point-weighted $R^2$ is not, because moving the split
changes how many points each segment holds. In practice that bias hands most points to
whichever segment fits better and pins the answer to the edge of the search range. On
synthetic spectra with a known break the residual criterion recovered 7.94 kHz against
a true 8.0 kHz where the weighted-$R^2$ criterion gave 7.57 kHz, and on the real VEST
spectrum in section 8.4 the weighted version sat exactly on its search boundary.

### 6.4 Frequency-band integrated power

Integrating the PSD over a band gives the fluctuation power in that band:

$$P_{[f_1,f_2]} = \int_{f_1}^{f_2} S(f)\,\mathrm{d}f.$$

`compute_band_power` integrates by trapezoid over caller-named bands and can form
caller-defined ratios such as high/low. Two conventions worth knowing:

- Band edges are **closed**, $[f_1, f_2]$: a PSD sample landing exactly on an edge
  belongs to the band, so two adjacent bands sharing an edge both include it. With a
  fine frequency grid this is a negligible effect; with a coarse one it is not, which
  is why it is specified rather than left to chance.
- A band holding fewer than two samples integrates to `0.0` rather than raising —
  a band narrower than $\Delta f$ has no trapezoid to form.

Band names carry no meaning to the library. `"low"`, `"high"`, `"mhd"` and
`"turbulent"` are labels you attach; nothing downstream interprets them.

### 6.5 Time-resolved spectra

A PSD assumes stationarity, which fails across an instability, a reconnection event or
a disruption. The short-time Fourier transform relaxes that to *local* stationarity by
sliding a window along the record:

$$x(t) \;\longrightarrow\; S(f, t).$$

The uncertainty relation between the two axes is unavoidable: a window of duration $T$
gives frequency resolution $\Delta f \approx 1/T$ and time resolution $\approx T$. A
2 ms window resolves ~500 Hz and blurs everything faster than 2 ms; a 0.2 ms window
tracks fast transients but cannot separate two modes 500 Hz apart.

`compute_spectrogram` takes the window either in samples (`nperseg`) or, more naturally
for physics, in seconds (`window_duration`), and returns physical time and frequency
coordinates with the time axis offset back onto the caller's own timebase. A signal
shorter than one window returns an empty but correctly shaped result — frequency axis
intact, zero time columns — rather than raising, so sweeping many channels of uneven
length is deterministic.

### 6.6 The derivative transfer function — why a Mirnov voltage is not a field

A magnetic pickup coil measures the flux change through its winding, so its output
voltage is proportional to a *time derivative*:

$$V(t) \propto \frac{\mathrm{d}B}{\mathrm{d}t}.$$

Differentiation multiplies each Fourier component by $i2\pi f$, so power spectra pick
up a factor $(2\pi f)^2$:

$$S_{\mathrm{d}B/\mathrm{d}t}(f) = (2\pi f)^2\, S_B(f).$$

If the field spectrum follows $S_B \propto f^{\alpha}$, the measured voltage spectrum
follows $f^{\alpha+2}$. **A spectral index fitted to raw pickup voltage is the field's
index plus two.** A field spectrum with $\alpha = -3$ presents as $-1$ in the raw
voltage; reporting that $-1$ as a magnetic slope is off by a factor of $f^2$ across the
whole band.

`compute_psd` therefore does **not** correct for this, deliberately. Hiding a
`dB/dt → B` integration inside a generic PSD routine would be wrong for every
diagnostic that is not a pickup coil, and invisible when it is wrong. The correction is
diagnostic-specific preparation and belongs upstream — section 7.2 does it explicitly
through the canonical VEST path, and section 8.3 verifies the $+2$ on real data.

## 7. The processing contract

### 7.1 What the input validation refuses

Every entry point runs the same time-axis check before touching the data, and it
**raises** rather than guessing. Three failures matter:

- **Non-monotonic or repeated timestamps** — the record is not a time series.
- **Length mismatch** between `time` and `data`.
- **Materially nonuniform sampling** — sample spacing varying by more than
  `NONUNIFORM_TOLERANCE` of the median step.

The third is the one worth dwelling on. Welch and the FFT assume a uniform grid; the
frequency axis is built as $f_k = k f_s / N$ from a single scalar $f_s$. Hand them a
jittered or gappy axis and they return a spectrum that *looks* perfectly normal and is
quantitatively wrong, with no indication anything happened. Silence is the dangerous
outcome here, so VAFT refuses the input and tells you to resample. It never
interpolates on your behalf, because the right interpolation for a fluctuation signal
is a physics decision.

In [ ]:
def show_refusal(label, call):
    """Run a deliberately invalid call and print the error it raises."""
    try:
        call()
    except ValueError as error:
        print(f"{label:24s} -> {error}")
    else:
        print(f"{label:24s} -> accepted (unexpected)")

fs_demo = 1.0e5
t_demo = np.arange(4096, dtype=float) / fs_demo
x_demo = np.random.default_rng(0).standard_normal(t_demo.size)

show_refusal("reversed time", lambda: compute_psd(t_demo[::-1], x_demo))

repeated = t_demo.copy()
repeated[10] = repeated[9]
show_refusal("repeated timestamp", lambda: compute_psd(repeated, x_demo))

jittered = np.sort(np.random.default_rng(1).uniform(0.0, t_demo[-1], t_demo.size))
show_refusal("nonuniform sampling", lambda: compute_psd(jittered, x_demo))

show_refusal("length mismatch", lambda: compute_psd(t_demo, x_demo[:-1]))

print(f"\ntolerance: sample spacing may vary by {NONUNIFORM_TOLERANCE:.2%} of the median step")

### 7.2 Preparing a physically defined signal

Section 6.6 established that a Mirnov voltage must be integrated before its spectrum
means anything about $B$. VAFT already has a canonical VEST path for that:
`vest_b_field_pol_probe_legacy` applies the low-pass filter, the per-channel
calibration, the integration and the baseline subtraction that the EFIT workflow uses.

One adjustment is needed. That path low-passes at 2.5 kHz, which is correct for
equilibrium reconstruction and destroys exactly the band we want here. The
configuration is a frozen dataclass, so `dataclasses.replace` raises the cutoff to the
native band while leaving the rest of the chain untouched — reusing the canonical
processing rather than writing a second, divergent one.

Two honest caveats about the result:

- The legacy **baseline window** is indexed for a full-acquisition record, not for the
  0.26–0.34 s window mapped here, so the *absolute* field offset is not meaningful in
  this sub-window.
- That does not affect the spectrum. Welch detrends each segment, so the fitted index
  is insensitive to the baseline choice — measured at $\Delta\alpha = 0.0008$ between
  the baseline-subtracted and raw-integrated variants.

In [ ]:
spectral_channel = outboard_channel
plasma_window = (0.305, 0.330)  # documented flat-top interval for shot 44740

voltage_time = np.asarray(
    get_path(ods, f"magnetics.b_field_pol_probe.{spectral_channel}.voltage.time")
)
voltage = np.asarray(
    get_path(ods, f"magnetics.b_field_pol_probe.{spectral_channel}.voltage.data")
)

# Per-channel calibration from the same table the mapper uses.
probe_definitions = [
    channel
    for channel in vest_equilibrium_magnetics_channel_definitions()
    if channel["kind"] == "b_field_pol_probe"
]
calibration = float(probe_definitions[spectral_channel]["calibration"])

# Native-bandwidth variant of the canonical VEST processing config.
fluctuation_config = replace(DEFAULT_VEST_MAGNETICS_PROCESSING, lowpass_cutoff=100_000.0)
field = vest_b_field_pol_probe_legacy(
    voltage_time, voltage, calibration, shot=44740, config=fluctuation_config
)

window = (voltage_time >= plasma_window[0]) & (voltage_time <= plasma_window[1])
analysis_time, analysis_field, analysis_voltage = (
    voltage_time[window],
    field[window],
    voltage[window],
)

sample_rate = 1.0 / np.median(np.diff(analysis_time))
print(f"channel {spectral_channel}: {probe_definitions[spectral_channel]['field_code']} "
      f"(calibration {calibration:g})")
print(f"{analysis_time.size} samples over {plasma_window[0]}-{plasma_window[1]} s "
      f"at {sample_rate:,.0f} Hz")
print(f"equilibrium cutoff {DEFAULT_VEST_MAGNETICS_PROCESSING.lowpass_cutoff:,.0f} Hz "
      f"-> fluctuation cutoff {fluctuation_config.lowpass_cutoff:,.0f} Hz")

### 7.3 Reproducibility

Nothing is chosen implicitly. `compute_psd` takes `window`, `nperseg`, `noverlap` and
`detrend` as explicit arguments, derives the sample rate from the time axis (unless you
override it), and records the parameters on the result. The returned
`FluctuationSpectrum` is a frozen dataclass, so a spectrum and the settings that
produced it travel together.

`units` is a free-text label the caller supplies. The library never infers units from
the data — it cannot know whether an array is tesla, volts or counts.

In [ ]:
baseline_spectrum = compute_psd(
    analysis_time,
    analysis_field,
    window="hann",
    nperseg=2048,
    noverlap=1024,
    detrend="constant",
    units="T**2/Hz",
)

print(f"method        {baseline_spectrum.method}")
print(f"sample_rate   {baseline_spectrum.sample_rate:,.0f} Hz")
print(f"units         {baseline_spectrum.units}")
print(f"frequencies   {baseline_spectrum.frequency.size} bins, "
      f"df = {baseline_spectrum.frequency[1]:.1f} Hz, "
      f"f_max = {baseline_spectrum.frequency[-1]:,.0f} Hz")

## 8. Spectral analysis of a VEST magnetic fluctuation

### 8.1 Composition

`analyze_fluctuation_spectrum` runs the PSD and every optional stage in one call: a fit
per caller-chosen interval, band powers with ratios, and a break. Every stage is
opt-in — with no `fit_ranges`, `bands` or break argument it returns exactly what
`compute_psd` returns.

Note the two $R^2$ values below. The low band is well described by a power law; the
high band is not, and the fit reports that rather than hiding it behind a confident
slope.

In [ ]:
low_band = (2.0e3, 1.0e4)
high_band = (1.5e4, 8.0e4)

result = analyze_fluctuation_spectrum(
    analysis_time,
    analysis_field,
    nperseg=2048,
    units="T**2/Hz",
    fit_ranges=[low_band, high_band],
    bands={"low": low_band, "high": high_band},
    ratios={"high_over_low": ("high", "low")},
    break_frequency=1.2e4,          # physics-informed: your boundary, not the library's
    break_fit_range=(2.0e3, 8.0e4),
)

low_fit, high_fit = result.fits
for name, band, fit in (("low", low_band, low_fit), ("high", high_band, high_fit)):
    print(f"{name:5s} {band[0]/1e3:5.1f}-{band[1]/1e3:5.1f} kHz  "
          f"alpha = {fit.alpha:+.2f} +/- {fit.stderr:.2f}  "
          f"R^2 = {fit.r_squared:.3f}  ({fit.n_points} points)")

print()
for name, value in result.band_power.items():
    print(f"{name:15s} {value:.4e}")

### 8.2 Normalization check, and when it does *not* hold

Section 6.1 claimed $\int S(f)\,\mathrm{d}f = \operatorname{var}(x)$. That identity is a
useful sanity check on a PSD estimator — it catches a wrong window normalization or a
mis-scaled sample rate — but it holds only for a **stationary** signal, and it is worth
seeing it fail.

The cell below checks it three ways:

1. on stationary broadband noise, where it holds to about a percent (the residual is
   Welch's own windowing and edge effects at this record length);
2. on the same noise plus a strong linear trend, where it fails badly;
3. on the real integrated field.

The real trace behaves like case 2, not case 1. Integrating a pickup voltage produces a
signal dominated by a slow ramp across the 25 ms window, and a 122 Hz frequency grid
cannot represent that ramp — most of `np.var` lives below the first frequency bin. The
same trace also carries a large DC offset from the baseline caveat in section 7.2, and
with `detrend=False` that offset contributes its *square* to the integral while
contributing nothing to the variance.

The practical lesson: use this identity to validate an estimator on a signal you know
is stationary, not to validate a measurement. For fitting a spectral index the
non-stationarity is harmless — Welch detrends each segment, and the fitted slope in
section 8.5 barely moves — but the integral no longer equals the variance.

In [ ]:
def parseval(label, time, values, **options):
    spectrum = compute_psd(time, values, nperseg=2048, **options)
    integrated = np.trapezoid(spectrum.psd, spectrum.frequency)
    variance = np.var(values)
    print(f"{label:34s} integral {integrated:.4e}  var {variance:.4e}  "
          f"error {abs(integrated - variance) / variance:8.2%}")

rng = np.random.default_rng(3)
stationary = 2.0 * rng.standard_normal(analysis_time.size)
trended = stationary + np.linspace(0.0, 50.0, analysis_time.size)

parseval("stationary noise", analysis_time, stationary, detrend=False)
parseval("the same, plus a linear trend", analysis_time, trended, detrend=False)
parseval("real integrated field", analysis_time, analysis_field, detrend=False)

print(f"\nfield mean {analysis_field.mean():.3e} T -> mean^2 = "
      f"{analysis_field.mean() ** 2:.3e}, against a variance of "
      f"{np.var(analysis_field):.3e}")

### 8.3 The $+2$ shift, measured

Section 6.6 predicted that the same window analysed as raw pickup voltage gives a
spectral index two higher than the integrated field. This is the check that stops a
Mirnov voltage spectrum from being reported as a magnetic-field spectrum.

The comparison uses the low band, well below Nyquist, because the discrete integration
and the anti-alias filter both roll off as the band edge is approached.

In [ ]:
voltage_spectrum = compute_psd(analysis_time, analysis_voltage, nperseg=2048)
voltage_fit = fit_power_law_spectrum(
    voltage_spectrum.frequency, voltage_spectrum.psd, f_range=low_band
)

print(f"alpha(B)        = {low_fit.alpha:+.2f}    <- integrated, calibrated field")
print(f"alpha(dB/dt)    = {voltage_fit.alpha:+.2f}    <- raw pickup voltage")
print(f"difference      = {voltage_fit.alpha - low_fit.alpha:+.2f}    (theory: +2)")

### 8.4 Spectral break: imposed versus data-driven

Both modes run on the same spectrum below. They answer different questions and can
legitimately disagree — the imposed break tests the boundary you brought, the search
reports where the data actually bends. Neither is identified with a plasma scale.

Passing both arguments, or neither, is an error rather than a silent default.

In [ ]:
imposed = result.spectral_break
search_range = (4.0e3, 4.0e4)
data_driven = find_spectral_break(
    result.frequency, result.psd, fit_range=(2.0e3, 8.0e4), search_range=search_range
)

for name, br in (("imposed", imposed), ("data-driven", data_driven)):
    print(f"{name:12s} mode={br.mode:8s} f_break={br.break_frequency/1e3:6.2f} kHz  "
          f"alpha_low={br.alpha_low:+.2f}  alpha_high={br.alpha_high:+.2f}  "
          f"R^2={br.r_squared:.3f}")

edge_low = data_driven.break_frequency <= search_range[0] * 1.05
edge_high = data_driven.break_frequency >= search_range[1] * 0.95
print(f"\nsearch range {search_range[0]/1e3:.0f}-{search_range[1]/1e3:.0f} kHz; "
      f"result sits on an edge: {edge_low or edge_high}")

try:
    find_spectral_break(result.frequency, result.psd, fit_range=(2.0e3, 8.0e4))
except ValueError as error:
    print(f"\nno mode given -> {error}")

### 8.5 Segment length: resolution against variance

The trade from section 6.1, measured. As `nperseg` grows, $\Delta f$ shrinks and finer
structure appears; at the same time fewer segments are averaged, so the estimate gets
noisier. Watch `R^2` fall as the spectrum becomes rougher while `alpha` stays put — the
underlying slope is real, but a single long segment measures it against a noisier
background.

The sweep stops at the record length: a segment longer than the signal is silently
shortened by SciPy, so the requested resolution would not be the one you got.

There is no universally correct choice. For a broadband power-law fit, favour more
averaging; for resolving a narrow coherent mode, favour resolution.

In [ ]:
print(f"{'nperseg':>8}  {'df [Hz]':>9}  {'segments':>9}  {'alpha':>7}  {'stderr':>7}  {'R^2':>6}")
for nperseg in (512, 1024, 2048, 4096):
    if nperseg > analysis_time.size:
        print(f"{nperseg:8d}  (longer than the {analysis_time.size}-sample record; skipped)")
        continue
    spectrum = compute_psd(analysis_time, analysis_field, nperseg=nperseg)
    fit = fit_power_law_spectrum(spectrum.frequency, spectrum.psd, f_range=low_band)
    segments = analysis_time.size / (nperseg * 0.5)   # scipy's default 50% overlap
    print(f"{nperseg:8d}  {spectrum.frequency[1]:9.1f}  {segments:9.1f}  "
          f"{fit.alpha:+7.2f}  {fit.stderr:7.3f}  {fit.r_squared:6.3f}")

### 8.6 Band powers and the shared-edge effect

Bands are named by the caller and integrated as closed intervals, $[f_1, f_2]$. What
that means in practice is worth measuring rather than assuming.

Splitting a band in two at an edge that lands **exactly on a frequency sample** is
lossless: the two halves sum to the whole, because a trapezoid sharing one endpoint
contributes no extra area there. Splitting at an edge that falls **between** samples
leaves the straddling interval in neither half, so the two halves sum to slightly
*less* than the whole. The size of the gap is set by $\Delta f$ and the local PSD
level, and the cell below measures both cases on the same spectrum.

This matters when band powers are compared across shots analysed with different
`nperseg`: put band edges on the frequency grid, or accept a gap of order one bin.

In [ ]:
split_off_grid = 1.0e4
split_on_grid = float(result.frequency[np.argmin(np.abs(result.frequency - split_off_grid))])
print(f"df = {result.frequency[1]:.2f} Hz; "
      f"nearest grid point to {split_off_grid:,.0f} Hz is {split_on_grid:,.2f} Hz")

for label, split in (("off-grid edge", split_off_grid), ("on-grid edge", split_on_grid)):
    powers = compute_band_power(
        result.frequency,
        result.psd,
        {"lower": (2.0e3, split), "upper": (split, 4.0e4), "whole": (2.0e3, 4.0e4)},
        ratios={"upper_over_lower": ("upper", "lower")},
    )
    gap = powers["whole"] - (powers["lower"] + powers["upper"])
    print(f"\n{label}: split at {split:,.2f} Hz")
    for name in ("lower", "upper", "whole", "upper_over_lower"):
        print(f"  {name:18s} {powers[name]:.6e}")
    print(f"  {'gap (whole - parts)':18s} {gap:+.3e}  ({gap / powers['whole']:+.4%})")

## 9. Plotting

### 9.1 The view-model contract

VAFT's renderers accept typed view models and nothing else. A renderer never sees an
ODS, a shot number or a file path, and never computes anything — passing it a data
object raises a `TypeError` naming the adapter to use instead. `PowerSpectrum` is the
model for a spectrum; `PowerSpectrum.from_result` builds one straight from a
`FluctuationSpectrum`, carrying the units label across into the axis label.

Fitted segments are *not* generated by the model. If you want them drawn you pass them,
so nothing appears on the plot that the caller did not compute.

In [ ]:
plain = PowerSpectrum.from_result(result, title=f"Channel {spectral_channel}, as computed")
print("y_label taken from result.units:", plain.y_label)
print("nothing drawn unless asked      :",
      f"fits={plain.fits}, reference_slopes={plain.reference_slopes}, "
      f"markers={plain.marker_frequencies}")

fig, ax = render_power_spectrum(plain, figsize=(6.5, 4))
plt.close(fig)

### 9.2 Reference slopes are yours

A reference slope is a straight line in log-log space with a slope you choose. VAFT
supplies no values: there are no built-in `-5/3`, `-8/3` or any other constants, and
the renderer attaches no physical meaning to a number. When a guide carries no label,
its legend entry is the bare exponent.

Three ways to specify one, all shown below:

- **derived from your own fit** — `ReferenceSlope(slope=low_fit.alpha, ...)`, so the
  guide shows what this shot actually did;
- **an arbitrary value** — `reference_slopes=[-1.5, -2.0]`, or any other number,
  labelled however you like;
- **explicitly anchored** — `anchor=(frequency, psd)` places the line through a chosen
  point, which is how you offset a guide off the data so it stays readable.

Without an anchor the guide passes through the measured PSD at the geometric-mean
frequency of the drawn range: a deterministic, purely numerical placement.

`marker_frequencies` adds labelled vertical lines the same way — a characteristic
frequency you brought, named by you.

In [ ]:
def fit_segment(fit, **style):
    """One drawn power-law segment from a SpectralFit."""
    edges = np.array(fit.frequency_range, dtype=float)
    return Series(
        x=edges,
        y=10.0**fit.intercept * edges**fit.alpha,
        label=f"fit {fit.alpha:+.2f} (R^2={fit.r_squared:.2f})",
        style={"linewidth": 2.0, **style},
    )

# An offset anchor: one decade above the PSD at 5 kHz, so the guide sits clear of the data.
anchor_frequency = 5.0e3
anchor_index = int(np.argmin(np.abs(result.frequency - anchor_frequency)))
offset_anchor = (float(result.frequency[anchor_index]), float(result.psd[anchor_index]) * 10.0)

spectrum_model = PowerSpectrum(
    frequency=result.frequency,
    psd=result.psd,
    fits=(fit_segment(low_fit, color="tab:red"), fit_segment(high_fit, color="tab:green")),
    reference_slopes=(
        # 1. derived from this shot's own fit
        ReferenceSlope(slope=low_fit.alpha, label=f"low-band fit ({low_fit.alpha:+.2f})"),
        # 2. an arbitrary caller value, unlabelled -> legend shows the bare exponent
        ReferenceSlope(slope=-2.0),
        # 3. the same slope, explicitly anchored a decade above the data
        ReferenceSlope(
            slope=-2.0,
            label="f^-2, offset for legibility",
            anchor=offset_anchor,
            style={"color": "tab:purple", "linestyle": ":"},
        ),
    ),
    marker_frequencies=((imposed.break_frequency, "imposed break"),),
    label=f"channel {spectral_channel} (B)",
    y_label=f"PSD [{result.units}]",
    title=f"Magnetic fluctuation PSD, shot 44740, {plasma_window[0]}-{plasma_window[1]} s",
    x_limits=(1.0e3, 1.0e5),
)

fig, ax = render_power_spectrum(spectrum_model, figsize=(7.5, 5))
fig.savefig(output_dir / "mirnov_power_spectrum.png", dpi=200, bbox_inches="tight")
fig

### 9.3 Generic spectrogram

`compute_spectrogram` is the diagnostic-independent counterpart to the MATLAB-parity
path in section 4: the same `(time, data)` contract, a window given in seconds, and
physical coordinates on the caller's own timebase.

In [ ]:
window_duration = 2.0e-3
spectrogram = compute_spectrogram(
    analysis_time, analysis_field, window_duration=window_duration, overlap=0.75
)

print(f"window {window_duration * 1e3:.1f} ms -> "
      f"df = {spectrogram.frequency[1]:.0f} Hz, {spectrogram.time.size} time columns")
print(f"time axis spans {spectrogram.time[0]:.4f}-{spectrogram.time[-1]:.4f} s "
      f"(signal: {analysis_time[0]:.4f}-{analysis_time[-1]:.4f} s)")

fig, ax = render_spectrogram(
    Spectrogram.from_result(
        spectrogram,
        max_frequency=8.0e4,
        value_label="|B| [T]",
        title=f"Channel {spectral_channel}, integrated field",
    ),
    figsize=(7.5, 4),
)
fig.savefig(output_dir / "mirnov_field_spectrogram.png", dpi=200, bbox_inches="tight")
fig

### 9.4 The same plot from an ODS in one call

Everything above went through the process layer by hand, which is what you want while
exploring. For a routine plot, the `vaft.omas` adapter does the extraction, the
analysis and the rendering in one call, and `reference_slopes` passes straight through.

The adapter analyses the signal **as stored** — for a Mirnov channel that is the
voltage, so this is a `dB/dt` spectrum and its index is the field's plus two, exactly
as section 8.3 measured. That is the honest default: the adapter does not silently
integrate.

In [ ]:
fig, ax = vaft.omas.plot_magnetics_spectrum_mirnov(
    ods,
    channel=spectral_channel,
    time_range=plasma_window,
    nperseg=2048,
    fit_ranges=[low_band],
    reference_slopes=[-1.0],
    series_label="raw voltage",
    title=f"Channel {spectral_channel} voltage spectrum (dB/dt, not B)",
)
ax.set_ylabel("PSD [V^2/Hz]")
fig.savefig(output_dir / "mirnov_voltage_spectrum_adapter.png", dpi=200, bbox_inches="tight")
fig

## 10. Diagnostic independence

This is what the whole separation is for. The routines used above have no diagnostic
parameter, so the same calls work on any scalar time series once someone has prepared a
physically defined signal.

To show that on real data rather than by assertion, the section below switches to a
completely different diagnostic: the VEST soft X-ray array. Different physics,
different detector, different digitizer, a different shot (45531) and roughly four times
the sample rate. The mapper writes brightness traces to the `soft_x_rays` IDS; the
spectral code is identical.

In [ ]:
from vaft.machine_mapping.soft_x_rays import soft_x_rays_from_digitizer_csv

sxr_ods = soft_x_rays_from_digitizer_csv(45531, 22577)
sxr_channel = 0
sxr_time = np.asarray(sxr_ods[f"soft_x_rays.channel.{sxr_channel}.brightness.time"]).ravel()
sxr_signal = np.asarray(sxr_ods[f"soft_x_rays.channel.{sxr_channel}.brightness.data"]).ravel()

print(f"{sxr_ods[f'soft_x_rays.channel.{sxr_channel}.name']}")
print(f"{sxr_signal.size} samples over {sxr_time[0]:.3f}-{sxr_time[-1]:.3f} s "
      f"at {1 / np.median(np.diff(sxr_time)):,.0f} Hz")
print(f"(magnetics above: {analysis_field.size} samples at {sample_rate:,.0f} Hz)")

In [ ]:
# Identical calls, a different diagnostic. Nothing here names soft X-rays.
sxr_result = analyze_fluctuation_spectrum(
    sxr_time,
    sxr_signal,
    nperseg=4096,
    units="a.u.**2/Hz",
    fit_ranges=[(1.0e4, 1.0e5)],
    bands={"low": (1.0e3, 1.0e4), "high": (1.0e4, 1.0e5)},
    ratios={"high_over_low": ("high", "low")},
)
sxr_fit = sxr_result.fits[0]

print(f"SXR 10-100 kHz: alpha = {sxr_fit.alpha:+.2f} +/- {sxr_fit.stderr:.2f}  "
      f"R^2 = {sxr_fit.r_squared:.3f}  ({sxr_fit.n_points} points)")
print(f"band power: {sxr_result.band_power}")

The adapter layer follows the same pattern: one registered renderer per domain over one
shared drawing body, so `soft_x_rays_spectrum` and `magnetics_spectrum_mirnov` differ
only in which IDS paths they read.

In [ ]:
print("spectrum plots available for this SXR ODS:")
for row in vaft.omas.available_plots(sxr_ods):
    if row["view"] in ("spectrum", "spectrogram"):
        print(f"  {row['name']}")

fig, ax = vaft.omas.plot_soft_x_rays_spectrum(
    sxr_ods,
    channel=sxr_channel,
    nperseg=4096,
    fit_ranges=[(1.0e4, 1.0e5)],
    reference_slopes=[ReferenceSlope(slope=sxr_fit.alpha, label=f"fitted {sxr_fit.alpha:+.2f}")],
)
fig.savefig(output_dir / "soft_x_ray_power_spectrum.png", dpi=200, bbox_inches="tight")
fig

In [ ]:
# Side by side: two diagnostics, one analysis path.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

render_power_spectrum(
    PowerSpectrum(
        frequency=result.frequency, psd=result.psd,
        fits=(fit_segment(low_fit, color="tab:red"),),
        label="Mirnov, integrated B",
        y_label=f"PSD [{result.units}]",
        title=f"Magnetics ch {spectral_channel}, shot 44740",
        x_limits=(1.0e3, 1.0e5),
    ),
    ax=axes[0],
)
render_power_spectrum(
    PowerSpectrum(
        frequency=sxr_result.frequency, psd=sxr_result.psd,
        fits=(fit_segment(sxr_fit, color="tab:red"),),
        label="SXR brightness",
        y_label=f"PSD [{sxr_result.units}]",
        title=f"Soft X-rays ch {sxr_channel}, shot 45531",
        x_limits=(1.0e3, 4.0e5),
    ),
    ax=axes[1],
)
fig.tight_layout()
fig.savefig(output_dir / "diagnostic_independence.png", dpi=200, bbox_inches="tight")
fig

## 11. Summary

- Spectral analysis lives in `vaft.process.fluctuation` and takes a plain
  `(time, data)` pair. It has no diagnostic parameter, and section 10 exercised it on
  two unrelated diagnostics with identical calls.
- Diagnostic-specific preparation stays upstream. For Mirnov coils that means
  integrating `dB/dt` to `B` through the canonical VEST path before any spectrum is
  computed — worth exactly 2 in spectral index, measured in section 8.3.
- The time-axis contract refuses non-monotonic and materially nonuniform input instead
  of returning a plausible, wrong frequency axis.
- Every frequency interval — fit ranges, bands, break boundaries, search ranges — is
  supplied by the caller. Fits report `r_squared` and `stderr` alongside `alpha`, so a
  band the power-law model does not describe says so.
- Break analysis keeps physics-informed and data-driven modes distinct, and scores
  candidates by total squared residual rather than a point-weighted `R^2`.
- Renderers consume typed view models and draw only what they are given. Reference
  slopes are caller-supplied values with optional labels and anchoring; VAFT ships none
  and interprets none.
- Derived spectral products are returned as typed results, never written into
  non-standard ODS paths.

## Open Implementation Tasks

- Multi-channel analysis — cross-spectrum, coherence, cross-phase and mode numbers —
  builds on this single-channel foundation and is tracked separately.
- Interferometer fluctuation spectra need the `.mat` line-density loader before the
  registered `interferometer_spectrum` renderer can be demonstrated on real data.
- `soft_x_rays_time_power` keeps its canonical name while reading `brightness`;
  renaming it is a breaking API change deferred to a dedicated migration.
- Wire these products into the Snakemake pipeline once the fluctuation stage is defined.